# SaaS Revenue Forecasting Example

This notebook demonstrates how to use Revenue Builder to create revenue forecasts for a SaaS company.

## 1. Setup and Import

In [ ]:
# Import Revenue Builder
from revenue_builder import RevenueModel
from revenue_builder.business_models import BusinessModelTemplates
from revenue_builder.visualization import Dashboard

import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Generate Sample Data

For this example, we'll generate sample SaaS data. In practice, you would load your own historical data.

In [ ]:
# Generate 36 months of sample SaaS data
sample_data = BusinessModelTemplates.generate_sample_data(
    business_type='saas',
    periods=36,
    start_date='2021-01-01'
)

# Display first few rows
print(f"Generated {len(sample_data)} months of data")
sample_data.head(10)

## 3. Initialize Revenue Model

In [ ]:
# Initialize SaaS revenue model
model = RevenueModel(business_type='saas')

print(model.summary())

## 4. Load and Validate Data

In [ ]:
# Load data
model.load_data(sample_data, validate=True, preprocess=True)

# Display processed data info
print(f"\nData shape: {model.processed_data.shape}")
print(f"Columns: {list(model.processed_data.columns)}")

## 5. Train Forecasting Models

In [ ]:
# Train multiple models
# Auto-select appropriate models for SaaS business
trained_models = model.train(auto_select=True)

print(f"\nTrained {len(trained_models)} models:")
for model_name in trained_models.keys():
    print(f"  - {model_name}")

## 6. Generate Revenue Forecast

In [ ]:
# Generate 24-month forecast
forecast = model.predict(periods=24, confidence_level=0.95)

# Display forecast
print("\nRevenue Forecast:")
forecast.head(12)

## 7. Calculate Key Metrics

In [ ]:
# Calculate SaaS metrics
metrics = model.calculate_metrics()

# Display key metrics
print("\nKey SaaS Metrics:")
print(f"Current MRR: ${metrics.get('current_mrr', 0):,.0f}")
print(f"Current ARR: ${metrics.get('current_arr', 0):,.0f}")
print(f"Churn Rate: {metrics.get('avg_churn_rate', 0):.1%}")
print(f"LTV:CAC Ratio: {metrics.get('ltv_cac_ratio', 0):.1f}x")
print(f"MoM Growth: {metrics.get('mom_growth_rate', 0):.1%}")

## 8. Visualize Forecast

In [ ]:
# Create dashboard
dashboard = Dashboard()

# Plot revenue forecast
fig = dashboard.plot_revenue_forecast(
    historical=sample_data,
    forecast=forecast,
    date_column='date',
    revenue_column='mrr',
    forecast_column='forecast',
    show_confidence=True,
    title='SaaS MRR Forecast'
)

plt.show()

## 9. Scenario Analysis

In [ ]:
# Run scenario analysis with different assumptions
scenarios = model.scenario_analysis(
    variables={
        'churn_rate': [0.02, 0.03, 0.05],  # Optimistic, base, pessimistic
        'new_customers': [60, 50, 40]       # Optimistic, base, pessimistic
    },
    monte_carlo=True,
    n_simulations=5000
)

print(f"\nGenerated {len(scenarios)} scenarios")

## 10. Compare Scenarios

In [ ]:
# Plot scenario comparison
scenario_forecast = {k: v for k, v in scenarios.items() if k != 'monte_carlo'}

fig = dashboard.plot_scenario_comparison(
    scenarios=scenario_forecast,
    revenue_column='forecast',
    title='Revenue Scenarios Comparison'
)

plt.show()

## 11. Monte Carlo Distribution

In [ ]:
# Plot Monte Carlo results
if 'monte_carlo' in scenarios:
    mc_results = scenarios['monte_carlo']
    
    fig = dashboard.plot_monte_carlo_distribution(
        results=mc_results,
        value_column='total_revenue',
        title='Monte Carlo Simulation - Revenue Distribution'
    )
    
    plt.show()

## 12. Export Results

In [ ]:
# Export comprehensive report
model.export_report(
    path='saas_revenue_forecast_2024.xlsx',
    include_visuals=True,
    include_commentary=True
)

print("\n✓ Report exported successfully!")

## 13. Model Performance

In [ ]:
# Get model performance metrics
performance = model.get_model_performance()

if not performance.empty:
    print("\nModel Performance:")
    print(performance[['model', 'rmse', 'mae', 'r2']].to_string(index=False))

## Summary

This notebook demonstrated:
1. Setting up a SaaS revenue forecasting model
2. Loading and validating data
3. Training multiple forecasting models
4. Generating revenue forecasts
5. Calculating key SaaS metrics
6. Creating visualizations
7. Running scenario analysis
8. Exporting comprehensive reports

Revenue Builder provides a complete solution for revenue forecasting with minimal code!